## 4 Подготовка данных к обучению

**Входы**

- Данные после очистки и разбиения
- Спецификации моделей из этапа 3.5
- Таблица выводов и решения по признакам

**Выходы**

- Правила обработки и готовые входы для моделей

**Что делаем**

Заполним пропуски и переведём категории в понятный модели вид. Все нужные для этого значения считаем только по train

In [ ]:
# dev-only: переходит в корень проекта; в сборку не входит
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


### 4.1 Состав признаков и выборки

Используем все 11 признаков из спецификаций этапа 3.5. Целевую переменную отделяем от входов модели

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

NUMERICAL_FEATURES = [
    'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
]
NOMINAL_FEATURES = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
ORDINAL_FEATURES = ['loan_grade']
GRADE_ORDER = list('ABCDEFG')
FEATURES = NUMERICAL_FEATURES + NOMINAL_FEATURES + ORDINAL_FEATURES


In [ ]:
from sklearn.base import clone
from threadpoolctl import threadpool_limits

# Ограничиваем число вычислительных потоков для воспроизводимых запусков.
_ml_thread_limit = threadpool_limits(limits=1)
assert set(FEATURES) == set(train.columns) - {'loan_status'}
assert set(NUMERICAL_FEATURES) == set(numerical_columns)
assert set(NOMINAL_FEATURES) == set(nominal_columns) - {'loan_status'}
assert set(ORDINAL_FEATURES) == set(ordinal_columns)
assert train.index.intersection(validation.index).empty
assert train.index.intersection(test.index).empty
assert validation.index.intersection(test.index).empty

X_train = train.loc[:, FEATURES].copy()
y_train = train['loan_status'].astype(int).copy()
X_validation = validation.loc[:, FEATURES].copy()
y_validation = validation['loan_status'].astype(int).copy()
split_summary = pd.DataFrame([
    {'Выборка': name, 'Записей': len(frame), 'Доля дефолтов': frame['loan_status'].mean()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]
]).set_index('Выборка')
display(split_summary.round(4))


### 4.2 Числовые признаки

Пропуски заполняем медианами train и добавляем индикаторы пропусков. Затем вычитаем медиану и делим на межквартильный размах train — это масштабирование RobustScaler

Задаём эти операции как цепочку Pipeline, которая обучается вместе с моделью

In [ ]:
def make_numerical_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
        ('scale', RobustScaler()),
    ])


In [ ]:
numerical_audit = make_numerical_preprocessor().fit(X_train[NUMERICAL_FEATURES])
display(pd.DataFrame({
    'Медиана для пропусков': numerical_audit.named_steps['impute'].statistics_,
}, index=NUMERICAL_FEATURES))


### 4.3 Номинальные признаки

Пропуски заполняем модой — самым частым значением train. Каждую категорию кодируем отдельным столбцом 0/1 — one-hot-кодированием. Порядок категорий не задаём

In [ ]:
def make_nominal_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])


In [ ]:
nominal_audit = make_nominal_preprocessor().fit(X_train[NOMINAL_FEATURES])
display(pd.DataFrame({
    'Мода для пропусков': nominal_audit.named_steps['impute'].statistics_,
    'Категорий': [len(values) for values in nominal_audit.named_steps['encode'].categories_],
}, index=NOMINAL_FEATURES))


### 4.4 Порядковый признак

Для loan_grade сохраняем порядок A–G, кодируя его числами 0–6. Пропуски заполняем модой train

Логистическая регрессия предполагает одинаковую прибавку к сумме вкладов признаков при переходе на одну категорию, деревья могут задавать разные границы разбиений

In [ ]:
def make_ordinal_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('encode', OrdinalEncoder(categories=[GRADE_ORDER])),
    ])


In [ ]:
display(pd.DataFrame({'Грейд': GRADE_ORDER, 'Код': range(len(GRADE_ORDER))}))


### 4.5 Общая обработка

Объединяем три цепочки. Каждый кандидат при обучении получит собственную копию обработки, параметры которой определяются только по train

Ниже проверяем результат на первых пяти строках. Ставку по грейду не восстанавливаем, долю кредита не пересчитываем

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ('numerical', make_numerical_preprocessor(), NUMERICAL_FEATURES),
        ('nominal', make_nominal_preprocessor(), NOMINAL_FEATURES),
        ('ordinal', make_ordinal_preprocessor(), ORDINAL_FEATURES),
    ], remainder='drop', verbose_feature_names_out=True)


In [ ]:
preprocessor_template = make_preprocessor()
audit_preprocessor = clone(preprocessor_template).fit(X_train)
preprocessing_preview = pd.DataFrame(
    audit_preprocessor.transform(X_train.head(5)),
    columns=audit_preprocessor.get_feature_names_out(), index=X_train.head(5).index,
)
display(preprocessing_preview.round(3))
